# Lab - Diagnose the Data Path Before the Model

Three tasks that chase a confident wrong answer to its real cause. The store shortened the return window from 30 days to 14; the model did not change, the answers come back as fast as before - and customers still hear 30.

| Task | What you learn |
| --- | --- |
| 1 | Log what retrieval returned and which version the index was built from |
| 2 | Check the index against its source: version, and sections that changed |
| 3 | Rebuild the index, and refuse to answer from a stale one |

**You do not need an API key.** This lab ships with a Claude simulator, so every cell
runs offline. The code you write is exactly the code you would write against the real
API - set `ANTHROPIC_API_KEY` at home and the same notebook calls Claude for real.

**You do not write code from scratch.** Each cell already holds the code, with blanks
marked `...` and a comment telling you what goes in each one.

## Setup

Run this cell first, before anything else. Click it, then press **Shift+Enter**.

In [ ]:
# --- Lab setup (provided - just run it) ---
import shopassist_lab
from shopassist_lab import check

# Everything below this line is ordinary Claude API code.
from dotenv import load_dotenv
from anthropic import Anthropic
import json
import math
import re
from collections import Counter
from datetime import date
from pathlib import Path

load_dotenv()
client = Anthropic()

# The model name lives in one constant, so a later update changes one line.
MODEL = "claude-sonnet-5-5"
LIVE = "policy_live.txt"   # the policy as the store publishes it today
QUESTION = "Can I still return a jacket delivered 20 days ago?"

# Every run starts from the index production has today (built last month).
if not Path(".index_production.json").exists():
    Path(".index_production.json").write_text(Path("index.json").read_text())
Path("index.json").write_text(Path(".index_production.json").read_text())

WORD = re.compile(r"[a-z0-9]+")


def words(text):
    return WORD.findall(text.lower())


class BM25:
    def __init__(self, chunks, k1=1.5, b=0.75):
        self.chunks = chunks
        self.docs = [Counter(words(c["text"])) for c in chunks]
        self.avg = sum(sum(d.values()) for d in self.docs) / len(self.docs)
        df = Counter(w for d in self.docs for w in d)
        n = len(self.docs)
        self.idf = {w: math.log(1 + (n - f + 0.5) / (f + 0.5)) for w, f in df.items()}
        self.k1, self.b = k1, b

    def search(self, query, k=3):
        def score(doc):
            norm = self.k1 * (1 - self.b + self.b * sum(doc.values()) / self.avg)
            return sum(self.idf.get(w, 0) * doc[w] * (self.k1 + 1) / (doc[w] + norm)
                       for w in set(words(query)))
        ranked = sorted(zip(self.chunks, self.docs), key=lambda cd: -score(cd[1]))
        return [chunk for chunk, _ in ranked[:k]]


def chunk_by_section(text):
    chunks = []
    for block in text.split("\n\n"):
        lines = block.strip().split("\n")
        if len(lines) >= 2:
            chunks.append({"id": lines[0].title(), "text": block.strip()})
    return chunks

---

## Task 1 - Log what retrieval gave the model

In [ ]:
# ============================================================
# TASK 1 - Log what retrieval gave the model
# ============================================================
#
# WHAT TO DO
#   Finish ask(): load the index, find the two best sections,
#   send them to Claude as policy excerpts. Return a record of
#   the answer, the sections retrieved, and the index header -
#   which policy version it was built from, and when. Ask the
#   customers' question.
#
# WHY IT MATTERS
#   The store shortened the return window from 30 days to 14,
#   and customers still hear 30. When a retrieval system gives a
#   confident wrong answer, the first question is not "is the
#   model broken" but "what did retrieval give it". A record per
#   answer is how you can tell.
#
# WHERE TO SEE IT IN THE LECTURE
#   "Second, the header of the index: which version of the
#   policy it was built from, and when".
#
# HOW TO DO IT
#   Three blanks:
#
#       index["chunks"]                 the sections in the index
#       [chunk["id"] for chunk in found]    what was retrieved
#       index["built_from"]             the version the index was built from
# ============================================================

try:
    # TODO: replace each ... below with the value named beside it
    def ask(question):
        index = json.load(open("index.json"))
        found = BM25(...).search(question, k=2)   # index["chunks"]
        excerpts = "\n\n".join(chunk["text"] for chunk in found)
        response = client.messages.create(
            model=MODEL, max_tokens=200, output_config={"effort": "low"},
            system="You are ShopAssist. Answer in one sentence from the policy excerpts.",
            messages=[{"role": "user", "content": f"<policy>\n{excerpts}\n</policy>\n\n{question}"}])
        return {"answer": response.content[0].text,
                "retrieved": ...,   # [chunk["id"] for chunk in found]
                "built_from": ..., "built_on": index["built_on"]}   # index["built_from"]

    record = ask(QUESTION)
    if ... in record.values():   # provided: a blank left above, said plainly
        raise TypeError("ellipsis")
except (TypeError,) as exc:   # provided: a blank left above, said plainly
    if "llipsis" in str(exc):
        raise ValueError("You still have ... above. Replace each one with the value "
                         "named beside it.") from None
    raise
for key, value in record.items():
    print(f"{key:10} {value}")

check("diag_log", record=record)

---

## Task 2 - Check the index against its source

In [ ]:
# ============================================================
# TASK 2 - Check the index against its source
# ============================================================
#
# WHAT TO DO
#   Write staleness(): compare the version the index was built
#   from with the first line of the live policy, and list the
#   indexed sections whose text no longer appears in the live
#   file.
#
# WHY IT MATTERS
#   The model, its version and the latency are unchanged; only
#   the content changed. So look at the data path: the
#   retrieved section is the right one, the return window - but
#   it was indexed from version 7, on the 28th of September.
#   Claude answered faithfully from last month's rule.
#
# WHERE TO SEE IT IN THE LECTURE
#   "The policy changed after that, and nobody rebuilt the
#   index".
#
# HOW TO DO IT
#   Three blanks:
#
#       live.split("\n")[0]       the live file's version line
#       index_version != live_version     built from something else
#       chunk["text"] not in live         indexed text that is gone
# ============================================================

try:
    # TODO: replace each ... below with the value named beside it
    def staleness():
        index = json.load(open("index.json"))
        live = open(LIVE).read()
        index_version = index["built_from"]
        live_version = ...   # live.split("\n")[0]
        return {"index_version": index_version, "live_version": live_version,
                "stale": ...,   # index_version != live_version
                "changed_sections": [chunk["id"] for chunk in index["chunks"]
                                     if ...]}   # chunk["text"] not in live

    report = staleness()
    if ... in report.values():   # provided: a blank left above, said plainly
        raise TypeError("ellipsis")
except (TypeError,) as exc:   # provided: a blank left above, said plainly
    if "llipsis" in str(exc):
        raise ValueError("You still have ... above. Replace each one with the value "
                         "named beside it.") from None
    raise
for key, value in report.items():
    print(f"{key:16} {value}")

check("diag_stale", report=report)

---

## Task 3 - Rebuild, and never answer from a stale index

In [ ]:
# ============================================================
# TASK 3 - Rebuild, and never answer from a stale index
# ============================================================
#
# WHAT TO DO
#   Write rebuild(): cut the live policy into sections and save a
#   new index with its version and today's date. Write
#   guarded_ask(): refuse to answer while the index is stale.
#   Ask once before the rebuild, rebuild, and ask again.
#
# WHY IT MATTERS
#   The fix for today is a rebuild. The fix for the next policy
#   change is a check in code: an answer from an index that does
#   not match its source is refused - the customer is routed to
#   a person, and the team is told to rebuild, instead of
#   hearing last month's rule with confidence.
#
# WHERE TO SEE IT IN THE LECTURE
#   "Rebuild the index whenever the source changes, and check
#   retrieval before you touch the prompt or the model".
#
# HOW TO DO IT
#   Three blanks:
#
#       chunk_by_section(text)      the live policy, as sections
#       str(date.today())           when this index was built
#       staleness()["stale"]        do not answer from a stale index
# ============================================================

try:
    # TODO: replace each ... below with the value named beside it
    def rebuild():
        text = open(LIVE).read()
        index = {"built_from": text.split("\n")[0], "built_on": ...,   # str(date.today())
                 "chunks": ...}   # chunk_by_section(text)
        json.dump(index, open("index.json", "w"), indent=1)
        return index["built_from"]
    
    def guarded_ask(question):
        if ...:   # staleness()["stale"]
            return {"answer": None, "error": "index is stale - rebuild it, then ask again"}
        return ask(question)

    before = guarded_ask(QUESTION)
    print("before:", before)
    print("rebuilt from", rebuild())
    after_rebuild = guarded_ask(QUESTION)
except (TypeError,) as exc:   # provided: a blank left above, said plainly
    if "llipsis" in str(exc):
        raise ValueError("You still have ... above. Replace each one with the value "
                         "named beside it.") from None
    raise
except NameError as exc:   # provided: an earlier task has not run yet
    raise ValueError("{} - run the cells above first, Tasks 1 to 2 included."
                     .format(exc)) from None
print("after: ", after_rebuild)

check("diag_rebuild", before=before, after=after_rebuild, index=json.load(open("index.json")))

---

## Done

Log what retrieval returns and which version of the documents the index was built from, rebuild the index whenever the source changes, refuse to answer from a stale one, and check retrieval before you touch the prompt or the model.